In [1]:
import os
import mne
import numpy as np
import pandas as pd
from scipy.signal import resample
import h5py
import json
import warnings
from collections import defaultdict
warnings.filterwarnings("ignore")

In [2]:
import mne
from mne.preprocessing import ICA
try:
    from mne_icalabel import label_components
except Exception:
    label_components = None

In [3]:
# multi-scale sampling rates
SAMPLE_RATE_LIST = [200, 100, 50]  # Hz

# fixed segment length in samples (timestamps)
SEG_LEN = 400  # number of time steps per segment
# overlap in samples (timestamps), not in seconds
OVERLAP = 200    # e.g. 200 means 50% overlap for SEG_LEN=400

assert 0 <= OVERLAP < SEG_LEN, "OVERLAP must be in [0, SEG_LEN)."

sub_folder_path = f"L{SEG_LEN}"
sub_folder_path

'L400'

In [4]:
# root dir
root = 'AD-Auditory/'
# participants file path
participants_path = os.path.join(root, 'participants.tsv')
participants = pd.read_csv(participants_path, sep='\t')
participants

,participant_id,Gender,Age,Group,MMSE
0,sub-01,Male,65,Normal,28
1,sub-02,Female,70,Mild AD,23
2,sub-03,Male,75,Normal,27
3,sub-04,Male,82,Mild AD,21
4,sub-05,Male,75,Mild AD,21
5,sub-06,Female,69,-,-
6,sub-07,Male,89,Mild AD,19
7,sub-08,Male,70,Normal,24
8,sub-09,Female,57,Normal,26
9,sub-10,Male,81,Normal,28


In [5]:
# label mapping: diagnosis code -> label id
label_map = {'Mild AD': 1, 'Moderate AD': 1, 'MCI': 2, 'Normal': 0, '-': -1}

# build subject info: "sub-XXX" -> (label, pid)
sub_info = {}  # key: subject name, value: (label, pid)
for row in participants.itertuples(index=False):
    sub_name = row[0]         # e.g. 'sub-001'
    diag_code = row[3]         # e.g. 'A' / 'F' / 'C'
    pid = int(sub_name.split('-')[1])
    label = label_map[diag_code]
    sub_info[sub_name] = (label, pid)

len(sub_info), list(sub_info.items())[:5]

(35,
 [('sub-01', (0, 1)),
  ('sub-02', (1, 2)),
  ('sub-03', (0, 3)),
  ('sub-04', (1, 4)),
  ('sub-05', (1, 5))])

## Features

In [6]:
def data_preprocessing(
    raw: mne.io.Raw,
    notch_freq: float = 60.0,
    l_freq: float = 0.5,
    h_freq: float = 40.0,
    verbose: bool = True,
):
    """
    Preprocessing steps ：
      2) Set Montage 
      3) 60 Hz Notch（before band pass）
      4) bandpass filter（default 0.5–40 Hz）
      5) interpolate bad channels（if do_bad_interp is True）
      6) re-reference to average
      7) ICA（在 1 Hz 高通的副本上拟合，自动剔除眼动/肌电等分量，需 mne-icalabel）
      8) downsample to 250 Hz
    """
        
    # 2. Set Montage
    raw.set_montage(mne.channels.make_standard_montage('standard_1005'))
    if verbose:
        print("✔ Step 2, Montage set: 'standard_1005'.")
        
    # 3. Notch（工频）
    if notch_freq is not None:
        raw.notch_filter(freqs=[notch_freq], picks="eeg", verbose=False)
        if verbose:
            print(f"✔ Step 3: Notch @ {notch_freq} Hz")
        
    # 4. Bandpass Filter (0.5–40 Hz)
    raw.filter(l_freq=l_freq, h_freq=h_freq, picks="eeg", verbose=False)
    if verbose:
        print(f"✔ Step 4: Band-pass {l_freq}–{h_freq} Hz")
        
    return raw

In [7]:
def resample_time_series(data, original_fs, target_fs):
    """
    Resample each channel independently.
    data: (T_raw, C)
    return: (T_new, C)
    """
    T, C = data.shape
    new_length = int(T * target_fs / original_fs)
    resampled_data = np.zeros((new_length, C), dtype=np.float32)
    for i in range(C):
        # resample one channel
        resampled_data[:, i] = resample(data[:, i], new_length)
    return resampled_data


def compute_step(seg_len, overlap):
    """
    Compute sliding step (in samples) given segment length and overlap.
    """
    step = seg_len - overlap
    if step <= 0:
        raise ValueError(f"Invalid overlap={overlap}: step <= 0.")
    return step


def compute_num_segments(num_samples, seg_len, step):
    """
    Compute how many segments can be extracted from a sequence
    with given segment length and step.
    """
    if num_samples < seg_len:
        return 0
    return 1 + (num_samples - seg_len) // step

## PASS 1: Find total number of segments across all subjects

In [8]:
subject_segment_counts = defaultdict(lambda: defaultdict(int))
N_total = 0
standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                     'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
n_channels = len(standard_channels)

# step (timestamps)
STEP = compute_step(SEG_LEN, OVERLAP)
print("SEG_LEN =", SEG_LEN, "OVERLAP =", OVERLAP, "STEP =", STEP)

# Test 
# deal with the matlab 7.3 file format
sub_id = 1
for sub in os.listdir(root):
    if 'sub-' in sub:
        sub_path = os.path.join(root, sub, 'eeg/')
        print(sub_path)
        print(f"[PASS 1] Subject: {sub}")
        for file in os.listdir(sub_path):
            if '.fdt' in file:
                print("Read .fdt file to load raw EEG data")
                fdt_file_path = os.path.join(sub_path, file)
                data = np.fromfile(fdt_file_path, dtype=np.float32).reshape(-1,19)
                # 250Hz, 19 monopolar channels, no 'T3' and 'T4', only 'T7' and 'T8', which are same as 'T3' and 'T4'
                print("Raw EEG data shape:", data.shape)   
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                    'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                info = mne.create_info(ch_names=standard_channels, sfreq=250, ch_types=['eeg'] * len(standard_channels))
                raw = mne.io.RawArray(data.T, info)
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
                for fs in SAMPLE_RATE_LIST:
                    # resampled length if each trial
                    T_res = int(T_raw * fs / original_fs)
                    # compute number of segments
                    n_seg = compute_num_segments(T_res, SEG_LEN, STEP)
                    subject_segment_counts[sub][fs] += n_seg
                    N_total += n_seg
                    print(f"fs={fs}Hz: T_res={T_res}, STEP={STEP}, n_seg={n_seg}")
                print("\n")           
        sub_id += 1
        print("------------End of Subject---------------")

print("\n===================================")
print("Total segments N_total =", N_total)
print("Channels =", n_channels)
print("===================================")

if N_total == 0:
    raise RuntimeError("No segments found. Please check SEG_LEN / OVERLAP.")

SEG_LEN = 400 OVERLAP = 200 STEP = 200
AD-Auditory/sub-01\eeg/
[PASS 1] Subject: sub-01
Read .fdt file to load raw EEG data
Raw EEG data shape: (87500, 19)
Creating RawArray with float64 data, n_channels=19, n_times=87500
    Range : 0 ... 87499 =      0.000 ...   349.996 secs
Ready.
fs=200Hz: T_res=70000, STEP=200, n_seg=349
fs=100Hz: T_res=35000, STEP=200, n_seg=174
fs=50Hz: T_res=17500, STEP=200, n_seg=86


------------End of Subject---------------
AD-Auditory/sub-02\eeg/
[PASS 1] Subject: sub-02
Read .fdt file to load raw EEG data
Raw EEG data shape: (87500, 19)
Creating RawArray with float64 data, n_channels=19, n_times=87500
    Range : 0 ... 87499 =      0.000 ...   349.996 secs
Ready.
fs=200Hz: T_res=70000, STEP=200, n_seg=349
fs=100Hz: T_res=35000, STEP=200, n_seg=174
fs=50Hz: T_res=17500, STEP=200, n_seg=86


------------End of Subject---------------
AD-Auditory/sub-03\eeg/
[PASS 1] Subject: sub-03
Read .fdt file to load raw EEG data
Raw EEG data shape: (87500, 19)
Creating R

In [9]:
output_root = os.path.join('Processed', sub_folder_path, 'AD-Auditory')
os.makedirs(output_root, exist_ok=True)

X_path = os.path.join(output_root, 'X.dat')
y_path = os.path.join(output_root, 'y.dat')
meta_path = os.path.join(output_root, 'meta.json')

print("X path:", X_path)
print("y path:", y_path)

# create memmap storage
X_mm = np.memmap(X_path, dtype='float32', mode='w+', shape=(N_total, SEG_LEN, n_channels))
y_mm = np.memmap(y_path, dtype='float32', mode='w+', shape=(N_total, 3))

X path: Processed\L400\AD-Auditory\X.dat
y path: Processed\L400\AD-Auditory\y.dat


In [10]:
cur = 0  # current index in memmap
total_seconds_all = 0  # used for total hours calculation
for sub in os.listdir(root):
    if 'sub-' in sub:
        label, pid = sub_info[sub]
        sub_path = os.path.join(root, sub, 'eeg')
        print(sub_path)
        if sub not in sub_info:
            continue
    
        total_seg_sub = sum(subject_segment_counts[sub][fs] for fs in SAMPLE_RATE_LIST)
        if total_seg_sub == 0:
            continue
        print(f"[PASS 2] Subject: {sub}, expected total segments={total_seg_sub}")
        for file in os.listdir(sub_path):
            if '.fdt' in file:
                print("Read .fdt file to load raw EEG data")
                fdt_file_path = os.path.join(sub_path, file)
                data = np.fromfile(fdt_file_path, dtype=np.float32).reshape(-1,19)
                # 250Hz, 19 monopolar channels, no 'T3' and 'T4', only 'T7' and 'T8', which are same as 'T3' and 'T4'
                print("Raw EEG data shape:", data.shape)   
                standard_channels = ['Fp1', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'T7', 'C3', 
                                    'Cz', 'C4', 'T8', 'P7', 'P3', 'Pz', 'P4', 'P8', 'O1', 'O2']
                info = mne.create_info(ch_names=standard_channels, sfreq=250, ch_types=['eeg'] * len(standard_channels))
                raw = mne.io.RawArray(data.T, info)
                raw = data_preprocessing(
                    raw=raw,
                    notch_freq=50,  # data collected in German, so notch at 50 Hz
                    l_freq=0.5,
                    h_freq=45.0,
                    verbose=True
                )
                original_fs = raw.info['sfreq']
                T_raw = raw.n_times
                data_raw = raw.get_data().T.astype('float32')  # (C, T) -> (T, C)
                for fs in SAMPLE_RATE_LIST:
                    # resample to target fs
                    data = resample_time_series(data_raw, original_fs, fs)
                    T_res, _ = data.shape
                    print(f"fs={fs}Hz: resampled shape={data.shape}")
        
                    # overlapped sliding window with fixed STEP (in timestamps)
                    starts = np.arange(0, T_res - SEG_LEN + 1, STEP, dtype=int)
                    print(f"fs={fs}Hz: segments={len(starts)}")
        
                    for s in starts:
                        if cur >= N_total:
                            raise RuntimeError("Exceeded predicted N_total.")
        
                        X_mm[cur] = data[s:s + SEG_LEN]   # (SEG_LEN, C)
                        y_mm[cur, 0] = float(label)       # label
                        y_mm[cur, 1] = float(pid)         # subject ID
                        y_mm[cur, 2] = float(fs)          # sample rate (scale)
                        cur += 1
                print("\n")           
        print("------------End of Subject---------------")

total_hours_all = total_seconds_all / 3600.0
print("DONE: cur =", cur, " expected N_total =", N_total)
print(f"Total hours across all subjects: {total_hours_all:.2f} hours")

# flush
del X_mm
del y_mm

# save meta
meta = {
    "N": int(N_total),
    "T": SEG_LEN,
    "C": int(n_channels),
    "SAMPLE_RATE_LIST": SAMPLE_RATE_LIST,
    "OVERLAP": int(OVERLAP),   # in samples
    "STEP": int(STEP),
    "X_path": X_path,
    "y_path": y_path,
}
with open(meta_path, "w") as f:
    json.dump(meta, f, indent=2)

print("Saved meta:", meta_path)

AD-Auditory/sub-01\eeg
[PASS 2] Subject: sub-01, expected total segments=609
Read .fdt file to load raw EEG data
Raw EEG data shape: (87500, 19)
Creating RawArray with float64 data, n_channels=19, n_times=87500
    Range : 0 ... 87499 =      0.000 ...   349.996 secs
Ready.
✔ Step 2, Montage set: 'standard_1005'.
✔ Step 3: Notch @ 50 Hz
✔ Step 4: Band-pass 0.5–45.0 Hz
fs=200Hz: resampled shape=(70000, 19)
fs=200Hz: segments=349
fs=100Hz: resampled shape=(35000, 19)
fs=100Hz: segments=174
fs=50Hz: resampled shape=(17500, 19)
fs=50Hz: segments=86


------------End of Subject---------------
AD-Auditory/sub-02\eeg
[PASS 2] Subject: sub-02, expected total segments=609
Read .fdt file to load raw EEG data
Raw EEG data shape: (87500, 19)
Creating RawArray with float64 data, n_channels=19, n_times=87500
    Range : 0 ... 87499 =      0.000 ...   349.996 secs
Ready.
✔ Step 2, Montage set: 'standard_1005'.
✔ Step 3: Notch @ 50 Hz
✔ Step 4: Band-pass 0.5–45.0 Hz
fs=200Hz: resampled shape=(70000, 19

## Load and check the processed data

In [11]:
import json
import numpy as np

# load meta information
meta_path = meta_path  # if you already have meta_path in current notebook
with open(meta_path, "r") as f:
    meta = json.load(f)

N = meta["N"]
T = meta["T"]          # SEG_LEN
C = meta["C"]
X_path = meta["X_path"]
y_path = meta["y_path"]

print("Meta:")
print("  N =", N)
print("  T =", T)
print("  C =", C)
print("  X_path =", X_path)
print("  y_path =", y_path)
print("-------------------------------------")

# open memmap
X_mm = np.memmap(X_path, dtype='float32', mode='r', shape=(N, T, C))
y_mm = np.memmap(y_path, dtype='float32', mode='r', shape=(N, 3))

# subject_id is stored in y[:, 1]
subject_ids = np.unique(y_mm[:, 1]).astype(int)

for sid in sorted(subject_ids):
    # find all indices for this subject
    idx = np.where(y_mm[:, 1] == sid)[0]

    # compute shapes logically (do not load all X into memory)
    n_seg = len(idx)
    x_shape = (n_seg, T, C)    # logical X shape for this subject
    y_shape = (n_seg, 3)       # logical y shape for this subject

    # get sampling rates for all segments of this subject
    fs_subject = y_mm[idx, 2]  # shape (n_seg,)


    print(f"Subject ID {sid:03d}: X shape={x_shape}, y shape={y_shape}")

# option: delete memmap views
del X_mm, y_mm

Meta:
  N = 32655
  T = 400
  C = 19
  X_path = Processed\L400\AD-Auditory\X.dat
  y_path = Processed\L400\AD-Auditory\y.dat
-------------------------------------
Subject ID 001: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 002: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 003: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 004: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 005: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 006: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 007: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 008: X shape=(609, 400, 19), y shape=(609, 3)
Subject ID 009: X shape=(1029, 400, 19), y shape=(1029, 3)
Subject ID 010: X shape=(1029, 400, 19), y shape=(1029, 3)
Subject ID 011: X shape=(1029, 400, 19), y shape=(1029, 3)
Subject ID 012: X shape=(1029, 400, 19), y shape=(1029, 3)
Subject ID 013: X shape=(1029, 400, 19), y shape=(1029, 3)
Subject ID 014: X shape=(1029, 400, 19), y shape=(1029, 3)
Subject ID 015: X shape=(10